In [ ]:
import os
import torch
import torch.nn as nn
import wandb
wandb.login(key="a7cd27db450c97d571343d5c6465b6f7a139989a")
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_from_disk

## Config

In [ ]:
# 自动检测是否在 Colab 环境
IS_COLAB = os.path.exists("/content")

# 基础路径配置
if IS_COLAB:
    # 建议把 Output 指向 Drive，防止断连丢失模型
    # 假设你已经 mount 了 drive 到 /content/drive
    BASE_OUTPUT_DIR = "/content/drive/MyDrive/Luna_Output"
else:
    BASE_OUTPUT_DIR = "./outputs"

# 确保目录存在
if not os.path.exists(BASE_OUTPUT_DIR):
    os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

# 你的其他配置
DATA_DIR = "./data"
CPT_DATA_PATH = os.path.join(DATA_DIR, "cpt_corpus.jsonl")
PROCESSED_DATA_PATH = os.path.join(DATA_DIR, "processed_cpt_dataset")

MODEL_ID = "Qwen/Qwen3-0.6B"
SEQ_LENGTH = 32768
SLIDING_WINDOW_SIZE = 4096

LORA_RANK = 16
LORA_ALPHA = 32

CPT_LEARNING_RATE = 1e-4

TTT_LEARNING_RATE = 1e-5
CHUNK_SIZE = 512
CLIP_THRESHOLD = 1.0

USE_TTT = True  # True = SWA+LoRA+TTT | False = SWA+LoRA

# === Environment Setup ===
os.environ["WANDB_PROJECT"] = "Luna-Project"
method_tag = "TTT" if USE_TTT else "NativeLoRA"
os.environ["WANDB_NAME"] = f"{method_tag}_CPT_qwen3_0.6B_2000steps"
# 隔离 Output Dir 以防止权重混淆
output_dir_name = "outputs_ttt_cpt" if USE_TTT else "outputs_lora_cpt"

## model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# === 辅助函数：联合裁剪 ===
def clip_tt_delta_(deltas, max_norm, norm_type=2.0):
    if isinstance(deltas, torch.Tensor):
        deltas = [deltas]
    norms_sq = [
        torch.sum(d.pow(norm_type), dim=(-2, -1), keepdim=True, dtype=torch.float32) 
        for d in deltas
    ]
    
    total_norm_sq = sum(norms_sq)
    total_norm = torch.pow(total_norm_sq, 1.0 / norm_type)
    
    clip_coef = max_norm / (total_norm + 1e-6)
    clip_coef = torch.clamp(clip_coef, max=1.0)
    
    if len(deltas) > 0:
        clip_coef = clip_coef.to(deltas[0].dtype)
    
    clipped_deltas = [d * clip_coef for d in deltas]
    return total_norm, clipped_deltas

class TTT_DownProj_Wrapper(nn.Module):
    def __init__(self, base_layer, embed_tokens=None):
        super().__init__()
        self.base_layer = base_layer
        self.h_dim = base_layer.out_features
        self.k_dim = base_layer.in_features
        
        # Init Params
        self.init_A = nn.Parameter(torch.randn(self.h_dim, LORA_RANK, dtype=torch.bfloat16) * 0.01)
        self.init_B = nn.Parameter(torch.randn(LORA_RANK, self.k_dim, dtype=torch.bfloat16) * 0.01)
        
        # Normalization & Projector
        self.ttt_norm = nn.LayerNorm(self.k_dim, dtype=torch.bfloat16)
        self.target_projector = nn.Linear(self.k_dim, self.h_dim, bias=False, dtype=torch.float32)
        nn.init.orthogonal_(self.target_projector.weight)
        self.target_projector.to(dtype=torch.bfloat16)
        
        self.scaling = LORA_ALPHA / LORA_RANK
        self.current_input_ids = None
        self.last_debug_metrics = {}

    def compute_fast_update(self, A, B, Z, V):
        projected_input = torch.matmul(Z, B.t()) 
        delta_A = torch.matmul(V.transpose(1, 2), projected_input)
        projected_error = torch.matmul(V, A)
        delta_B = torch.matmul(projected_error.transpose(1, 2), Z)
        return delta_A, delta_B

    def forward(self, x):
        base_out = self.base_layer(x)
        
        if not self.training and self.current_input_ids is None:
            lora_out = (x @ self.init_B.t()) @ self.init_A.t()
            return base_out + lora_out * self.scaling

        # === TTT Logic ===
        with torch.no_grad():
            x_detached = x.detach() 
            x_normed = self.ttt_norm(x_detached) # LayerNorm
            shifted_x = F.pad(x_normed[:, 1:, :], (0, 0, 0, 1))

        V = self.target_projector(shifted_x)

        # Prepare Chunks
        B_sz, Seq, _ = x.shape
        remainder = Seq % CHUNK_SIZE
        if remainder != 0:
            pad_len = CHUNK_SIZE - remainder
            Z_padded = F.pad(x, (0, 0, 0, pad_len))
            V_padded = F.pad(V, (0, 0, 0, pad_len))
        else:
            Z_padded, V_padded = x, V
            pad_len = 0
            
        new_seq_len = Z_padded.shape[1]
        num_chunks = new_seq_len // CHUNK_SIZE
        Z_chunks = Z_padded.view(B_sz * num_chunks, CHUNK_SIZE, self.k_dim)
        V_chunks = V_padded.view(B_sz * num_chunks, CHUNK_SIZE, self.h_dim)

        # Compute Gradients (Batch * Chunks 并行计算)
        dA_all, dB_all = self.compute_fast_update(
            self.init_A, self.init_B, Z_chunks, V_chunks
        )
        
        # === [Optimization] Vectorized Scan ===
        dA_view = dA_all.view(B_sz, num_chunks, self.h_dim, LORA_RANK)
        dB_view = dB_all.view(B_sz, num_chunks, LORA_RANK, self.k_dim)
        
        # 1. 计算 Inclusive Cumsum (当前 chunk 包含自己的梯度)
        # dim=1 是 chunks 维度
        acc_dA_inclusive = torch.cumsum(dA_view, dim=1)
        acc_dB_inclusive = torch.cumsum(dB_view, dim=1)
        
        # 2. 构造 Shift 后的 Exclusive Scan (当前 chunk 只包含之前的梯度)
        # 构造全零的初始状态 (对应第0个 chunk 没有任何历史梯度)
        zeros_A = torch.zeros(B_sz, 1, self.h_dim, LORA_RANK, device=dA_view.device, dtype=dA_view.dtype)
        zeros_B = torch.zeros(B_sz, 1, LORA_RANK, self.k_dim, device=dB_view.device, dtype=dB_view.dtype)
        
        # 拼接: [0, cumsum_0, cumsum_1, ..., cumsum_{N-2}]
        # 我们丢弃最后一个 cumsum (cumsum_{N-1})，因为它对后续没有贡献
        acc_dA = torch.cat([zeros_A, acc_dA_inclusive[:, :-1, ...]], dim=1)
        acc_dB = torch.cat([zeros_B, acc_dB_inclusive[:, :-1, ...]], dim=1)
        
        # Apply Updates
        lr = TTT_LEARNING_RATE * self.scaling
        acc_dA_flat = acc_dA.view(-1, self.h_dim, LORA_RANK)
        acc_dB_flat = acc_dB.view(-1, LORA_RANK, self.k_dim)
        
        raw_delta_A = lr * acc_dA_flat
        raw_delta_B = lr * acc_dB_flat
        
        # 计算 total_norm (修复了 numel > 1 的问题)
        # 注意：这里 raw_delta 包含了 batch * chunks 个梯度
        total_norm, [final_delta_A, final_delta_B] = clip_tt_delta_(
            [raw_delta_A, raw_delta_B], 
            max_norm=CLIP_THRESHOLD
        )
        
        # Apply Delta
        # final_delta_A 形状: [Batch * Chunks, H, R]
        # init_A 形状: [H, R] -> 需要广播到每个 Chunk
        init_A_exp = self.init_A.unsqueeze(0).expand(acc_dA_flat.shape[0], -1, -1)
        init_B_exp = self.init_B.unsqueeze(0).expand(acc_dB_flat.shape[0], -1, -1)
        
        A_eff = init_A_exp - final_delta_A
        B_eff = init_B_exp - final_delta_B
        
        # Parallel Computation of Output (所有 chunk 同时算)
        mid = torch.bmm(Z_chunks, B_eff.transpose(1, 2))
        lora_out_flat = torch.bmm(mid, A_eff.transpose(1, 2))
        
        lora_out = lora_out_flat.view(B_sz, new_seq_len, self.h_dim)
        if pad_len > 0:
            lora_out = lora_out[:, :Seq, :]

        if self.training:
            # 修复 total_norm 转 scalar 的问题
            if isinstance(total_norm, torch.Tensor) and total_norm.numel() > 1:
                scalar_norm = total_norm.mean().item()
            elif isinstance(total_norm, torch.Tensor):
                scalar_norm = total_norm.item()
            else:
                scalar_norm = total_norm

            self.last_debug_metrics = {
                "ttt/V_norm_avg": V.norm(p=2, dim=-1).mean().item(),
                "ttt/delta_norm_total": scalar_norm,  
                "ttt/clip_ratio": (total_norm > CLIP_THRESHOLD).float().mean().item() if isinstance(total_norm, torch.Tensor) else 0.0
            }
    
        return base_out + lora_out * self.scaling

## Data loading

In [ ]:
# 检查是否存在预处理好的数据文件夹
if os.path.exists(PROCESSED_DATA_PATH):
    print(f"🚀 发现预处理数据集: {PROCESSED_DATA_PATH}，直接加载！(跳过 Tokenizing)")
    dataset = load_from_disk(PROCESSED_DATA_PATH)
    is_pre_packed = True # 标记：数据已经是打包好的了
else:
    print(f"🐢 未发现预处理数据，加载原始 JSONL: {CPT_DATA_PATH} (需要在线处理)")
    dataset = load_dataset("json", data_files=CPT_DATA_PATH, split="train")
    is_pre_packed = False

print(f"Dataset Loaded. Samples: {len(dataset)}")

## Callback

In [ ]:
class TTTDebugCallback(TrainerCallback):
    """
    专门用于诊断 TTT 模块是否'死亡' (Collapse) 的回调函数。
    """
    def on_log(self, args, state, control, model=None, logs=None, **kwargs):
        if model is None: return
        
        ttt_metrics = {
            "ttt/avg_scale": [],
            "ttt/avg_delta_norm": [],
            "ttt/avg_V_norm": [],
            "ttt/projector_grad_norm": []
        }
        
        found_ttt = False
        # 遍历所有模块寻找 TTT Wrapper
        for name, module in model.named_modules():
            if isinstance(module, TTT_DownProj_Wrapper):
                found_ttt = True
                # 1. 读取 forward 期间缓存的运行时指标
                if hasattr(module, 'last_debug_metrics') and module.last_debug_metrics:
                    for k, v in module.last_debug_metrics.items():
                        if "delta_norm" in k: ttt_metrics["ttt/avg_delta_norm"].append(v)
                        if "V_norm" in k: ttt_metrics["ttt/avg_V_norm"].append(v)
                        if "clip_ratio" in k: 
                            # 如果你需要监控 clip_ratio，也可以加进去，这里演示加个 key
                            if "ttt/avg_clip_ratio" not in ttt_metrics: ttt_metrics["ttt/avg_clip_ratio"] = []
                            ttt_metrics["ttt/avg_clip_ratio"].append(v)
                
                # 2. 检查梯度流
                if hasattr(module, 'target_projector') and module.target_projector.weight.grad is not None:
                    grad_norm = module.target_projector.weight.grad.norm().item()
                    ttt_metrics["ttt/projector_grad_norm"].append(grad_norm)
        
        if not found_ttt:
            return

        # 计算平均值
        log_dict = {}
        for k, v_list in ttt_metrics.items():
            if len(v_list) > 0:
                log_dict[k] = sum(v_list) / len(v_list)
        
        # === 关键修改 ===
        # 不要手动传 step，防止与 Trainer 冲突
        if wandb.run is not None:
            wandb.log(log_dict)

class TTTSaveCallback(TrainerCallback):
    """
    HuggingFace Trainer 默认只保存 Adapter (LoRA) 和 Base Model。
    我们需要这个 Callback 在每次保存 Checkpoint 时，强制把 TTT 的自定义参数也存下来。
    """
    def on_save(self, args, state, control, **kwargs):
        checkpoint_folder = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        model = kwargs['model']
        
        # 提取 TTT 专属参数
        ttt_state_dict = {}
        for name, param in model.named_parameters():
            # 过滤条件：匹配你的 TTT Wrapper 中的参数名
            if any(k in name for k in ["target_projector", "init_A", "init_B", "ttt_layer"]):
                ttt_state_dict[name] = param.cpu()
        
        if len(ttt_state_dict) > 0:
            save_path = os.path.join(checkpoint_folder, "ttt_weights.pt")
            torch.save(ttt_state_dict, save_path)
            print(f"\n[TTT Callback] 已额外保存 {len(ttt_state_dict)} 个 TTT 张量至 {save_path} 喵。")

class TokenTrackingCallback(TrainerCallback):
    """
    绕过 Trainer 的 logging 机制，直接向 wandb 后端发送数据。
    """
    def __init__(self, seq_length):
        self.seq_length = seq_length

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % args.logging_steps == 0:
            tokens_per_step = (
                args.per_device_train_batch_size * args.gradient_accumulation_steps * args.world_size * self.seq_length
            )
            current_tokens = state.global_step * tokens_per_step
            
            if wandb.run is not None:
                wandb.log({
                    "train/tokens_trained": current_tokens,
                    "global_step": state.global_step
                })

## 模型初始化

In [ ]:
print(f"初始化 CPT 训练 ({method_tag})，窗口大小: {SLIDING_WINDOW_SIZE} 喵...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
    attn_implementation="flash_attention_2",
)
# [Fix] Qwen 默认没有 pad_token，这会导致 SFTTrainer 的 packing 逻辑失效或回退到 1024
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    # 某些版本的 TRL/Unsloth 需要明确更新 pad_token_id
    tokenizer.pad_token_id = tokenizer.eos_token_id
# [Fix] 显式告诉 Tokenizer 它的最大长度，防止 SFTTrainer 读取 config.json 中的旧值
tokenizer.model_max_length = SEQ_LENGTH
# ==========================================
# 2. 启用 Sliding Window Attention (SWA)
# ==========================================
if hasattr(model.config, "sliding_window"):
    model.config.sliding_window = SLIDING_WINDOW_SIZE
else:
    model.config.sliding_window = SLIDING_WINDOW_SIZE
    print(f"警告：该架构默认不显示 SWA 属性，已强制注入 config 喵。")

## 配置LoRA

In [ ]:
# ==========================================
# 3. 动态配置 LoRA
# ==========================================
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj"]

if not USE_TTT:
    target_modules.append("down_proj")
    print("Flag check: 使用原生 LoRA，已将 down_proj 加入训练目标 喵。")
else:
    print("Flag check: 使用 TTT 模式，将跳过 down_proj 的标准 LoRA 注入 喵。")

print(f"最终 LoRA 目标模块: {target_modules} 喵...")

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = target_modules,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## TTT module

In [ ]:
ttt_modules = [] # 初始化为空列表，防止后面引用报错

if USE_TTT:
    print("Flag check: 正在执行 TTT 手术替换 down_proj 喵...")
    embed_tokens = model.model.model.embed_tokens
    device = model.device 

    for i, layer in enumerate(model.model.model.layers):
        original_down = layer.mlp.down_proj
        
        # 创建 Wrapper
        wrapper = TTT_DownProj_Wrapper(original_down, embed_tokens)
        wrapper.to(device) 
        
        # 替换层
        layer.mlp.down_proj = wrapper
        ttt_modules.append(wrapper)

    print(f"所有 {len(ttt_modules)} 个 TTT 层已就位 喵！")

    # === 5. 注册 Hook (Input Capture) ===
    # 只有 TTT 需要捕获 input_ids
    def input_capture_hook(module, args, kwargs):
        input_ids = None
        if 'input_ids' in kwargs:
            input_ids = kwargs['input_ids']
        elif len(args) > 0:
            if isinstance(args[0], dict) and 'input_ids' in args[0]:
                input_ids = args[0]['input_ids']
            elif isinstance(args[0], torch.Tensor):
                input_ids = args[0]
                
        if input_ids is not None:
            # 这里的 ttt_modules 引用的是外部变量
            for mod in ttt_modules:
                mod.current_input_ids = input_ids

    model.register_forward_pre_hook(input_capture_hook, with_kwargs=True)
    print("TTT Input Capture Hook 已注册 喵。")
else:
    print("Flag check: 跳过 TTT 手术与 Hook 注册 喵。")

if USE_TTT:
    # 开启 TTT 参数的梯度
    for mod in ttt_modules:
        mod.init_A.requires_grad = True
        mod.init_B.requires_grad = True
        if hasattr(mod, 'target_projector'):
            mod.target_projector.weight.requires_grad = True

# 验证可训练参数
print("\n=== 可训练参数确认 ===")
trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
print(f"Total trainable tensors: {len(trainable_names)}")
if USE_TTT and any("target_projector" in n for n in trainable_names):
    print("状态: TTT Meta-Network 梯度正常开启 喵！")


## Trainer配置

In [ ]:
# ==========================================
# 7. Trainer 配置
# ==========================================
token_callback = TokenTrackingCallback(seq_length=SEQ_LENGTH)
num_proc = os.cpu_count()

if num_proc is None: num_proc = 4
# [Logic Switch] 
# 如果加载的是预处理数据 (is_pre_packed=True)，则 packing=False (因为已经 pack 过了)
# 如果加载的是原始数据 (is_pre_packed=False)，则 packing=True (让 Trainer 去做)
should_pack = not is_pre_packed

sft_config = SFTConfig(
    output_dir = output_dir_name,
    max_seq_length = SEQ_LENGTH,
    packing = should_pack, 
    dataset_text_field = "text" if should_pack else None,
    dataset_num_proc = num_proc,

    per_device_train_batch_size = 1,  
    gradient_accumulation_steps = 4,
    
    warmup_ratio = 0.05,
    num_train_epochs = 1,           # 跑满整个数据集 1 遍 喵
    learning_rate = CPT_LEARNING_RATE,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_torch",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    save_strategy = "steps",
    save_steps = 200,
    report_to = "wandb",
    remove_unused_columns = True, 
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = sft_config, # 将 SFTConfig 传入 args
    callbacks=[TTTSaveCallback, token_callback, TTTDebugCallback] if USE_TTT else [token_callback],
)

## 训练

In [ ]:
print(f"开始 CPT 训练, 模式: {method_tag} 喵！")

last_checkpoint = get_last_checkpoint(output_dir_name)

if last_checkpoint:
    print(f"发现历史 Checkpoint: {last_checkpoint}")
    # === 关键修复: 手动加载 TTT 权重 ===
    if USE_TTT:
        ttt_weights_path = os.path.join(last_checkpoint, "ttt_weights.pt")
        if os.path.exists(ttt_weights_path):
            print(f"正在恢复 TTT 状态 from: {ttt_weights_path}...")
            # map_location='cpu' 防止显存激增，load_state_dict 会自动处理 device
            ttt_state_dict = torch.load(ttt_weights_path, map_location='cpu')
            
            # strict=False 是必须的，因为 model 包含 LoRA 和 Base weights，而 ttt_state_dict 只有 TTT 部分
            keys = model.load_state_dict(ttt_state_dict, strict=False)
            print(f"TTT 权重恢复成功。Unexpected keys: {len(keys.unexpected_keys)}")
        else:
            print("【严重警告】找到 Checkpoint 但缺失 ttt_weights.pt！TTT 层将重置为随机初始化！")
    trainer.train(resume_from_checkpoint=True)
else:
    print("未找到 Checkpoint，开始全新训练。")
    print("\n=== [DEBUG] 检查 DataLoader 输出长度 ===")
    # 模拟 Trainer 的 DataLoader
    train_dataloader = trainer.get_train_dataloader()
    # 获取第一个 Batch
    for batch in train_dataloader:
        input_ids = batch['input_ids']
        print(f"实际输入 Batch Shape: {input_ids.shape}")
        print(f"实际输入 Sequence Length: {input_ids.shape[1]}")
        print(f"Trainer Config Max Length: {trainer.args.max_seq_length if hasattr(trainer.args, 'max_seq_length') else 'Unknown'}")
        # 或者检查 processing_class (Tokenizer)
        print(f"Trainer Tokenizer Max Length: {trainer.processing_class.model_max_length}")
        if input_ids.shape[1] < 2048:
            print("❌ 警告：数据依然很短！Packing 没有生效！")
            print("可能原因：Tokenizer 限制、数据源本身过短且不可拼接、或 Unsloth 优化冲突。")
        else:
            print("✅ 数据长度正常，看起来已经 Pack 好了。")
        break
    print("========================================\n")
    trainer.train()

## 保存

In [ ]:
print(f"\n=== 训练结束，正在保存 ===")
save_name = "model_cpt_ttt_final" if USE_TTT else "model_cpt_lora_final"
model.save_pretrained(save_name)
tokenizer.save_pretrained(save_name)
if USE_TTT:
    final_ttt_path = os.path.join(save_name, "ttt_weights.pt")
    ttt_state_dict = {}
    for name, param in model.named_parameters():
        if any(k in name for k in ["target_projector", "init_A", "init_B", "ttt_layer"]):
            ttt_state_dict[name] = param.cpu()
    torch.save(ttt_state_dict, final_ttt_path)
    print(f"TTT 最终权重已独立保存至: {final_ttt_path}")
print(f"所有工作完成！模型位于: {save_name} 喵！")